# MDR-TS v9.4
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v9.4
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/base_1.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

- Improving the `Expert B` model

## 0. Imports

In [1]:
import os
import random
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML / Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# Gradient Boosting (baseline model)
from xgboost import XGBRegressor

# PyTorch
import torch

# SciPy
from scipy.special import expit  # sigmoid

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

imports loaded
using: cpu


## 1. Environment Setup

In [2]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [3]:
# Project paths
VERSION = "v9"
SUBVERSION = "v9.4"
RUN_NAME = "mdr_ts_v9_4"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v9/v9.4

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [4]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_3.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_3.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_3.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

DROP_COLS = ["slope", "elev"]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=cols_present, inplace=True)
    print(f"{name}: dropped columns {cols_present}")
    print(f"\n{name}: shape={d.shape}")
    print(f"{name}: columns={len(d.columns)}")

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/train_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/val_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/test_derived_new.csv
train: dropped columns ['slope', 'elev']

train: shape=(16972, 352)
train: columns=352
val: dropped columns ['slope', 'elev']

val: shape=(2919, 352)
val: columns=352
test: dropped columns ['slope', 'elev']

test: shape=(2829, 352)
test: columns=352


In [5]:
# Dropping TouchNet because it has no rows in the validation split

DROP_STATION = "Touchet_WA_824"

def drop_station(df, station_id=DROP_STATION):
    before = len(df)
    out = df[df["station_id"] != station_id].copy()
    after = len(out)
    print(f"Dropped {station_id}: {before} -> {after} rows (-{before-after})")
    return out

train_df = drop_station(train_df)
val_df   = drop_station(val_df)
test_df  = drop_station(test_df)

Dropped Touchet_WA_824: 16972 -> 13661 rows (-3311)
Dropped Touchet_WA_824: 2919 -> 2919 rows (-0)
Dropped Touchet_WA_824: 2829 -> 2659 rows (-170)


In [6]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 352

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'aspect', 'DOY', 'soil_moisture_5cm', 'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio', 'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1', 'A_d_G_API_kobs2', 'A_d_G_API_kobs5', 'A_d_G_API_kobs7', 'A_d_G_API_kobs14']


## 4. Data Sanity Checks

In [7]:
# Column definitions (baseline)
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
  "precip_mm",

  "G_rain_sum_3d",
  "G_rain_sum_7d",
  "G_rain_sum_30d",
  "G_API",

  "G_DSLR",

  "C_lag_E_SAR_diff_kobs12",
  "C_lag_E_SAR_diff_kobs30",
  "C_lag_E_SAR_ratio_kobs30",
  "C_lag_F_NDVI_kobs30",
  "C_lag_LST_modis_kobs12",
  "C_lag_LST_modis_kobs30",
  "DOY",
  "D_sa_E_SAR_ratio",
  "D_sa_F_NDMI",
  "D_z_E_SAR_ratio",
  "D_z_F_NDMI",
  "E_SAR_diff",
  "E_SAR_ratio",
  "F_MSI",
  "F_NDMI",
  "V_ema_LST_modis_kobs30",
  "V_rollmax_E_SAR_diff_kobs14",
  "V_rollmax_E_SAR_diff_kobs30",
  "V_rollmax_F_NDVI_kobs30",
  "V_rollmax_G_API_kobs30",
  "V_rollmax_G_API_kobs7",
  "V_rollmax_LST_modis_kobs7",
  "V_rollmax_s2_b11_kobs30",
  "V_rollmean_G_API_kobs30",
  "V_rollmin_E_SAR_diff_kobs30",
  "V_rollmin_E_SAR_ratio_kobs30",
  "V_rollmin_F_NDMI_kobs30",
  "V_rollmin_G_API_kobs7",
  "V_rollmin_s2_b11_kobs30",
  "V_rollmin_s2_b12_kobs30",
  "s1_vh",
  "s2_b8",
  "A_d_LST_modis_kobs7",
  "A_grad_LST_modis_kobs14",
  "C_lag_E_SAR_diff_kobs6",
  "V_rollmax_E_SAR_diff_kobs7",
  "V_rollmax_E_SAR_ratio_kobs14",
  "V_rollmax_F_NDVI_kobs14",
  "s2_b12",
  "D_sa_LST_modis"
]

# quick validation
expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

Columns locked
  Features: 46
  Target:   soil_moisture_5cm


In [8]:
for d in (train_df, val_df, test_df):
    if "DOY" not in d.columns:
        d["DOY"] = pd.to_datetime(d["date"]).dt.dayofyear

def _season_bin(doy: np.ndarray) -> np.ndarray:
    doy = doy.astype(int)
    winter   = (doy <= 90) | (doy >= 335)
    shoulder = ((doy > 90) & (doy <= 150)) | ((doy >= 275) & (doy < 335))
    out = np.full_like(doy, 2)      # default summer=2
    out[shoulder] = 1
    out[winter]   = 0
    return out

for d in (train_df, val_df, test_df):
    d["season_bin"] = _season_bin(d["DOY"].to_numpy())

print("Season regimes ready (0=winter, 1=shoulder, 2=summer)")
print("  train counts:", train_df["season_bin"].value_counts().sort_index().to_dict())
print("  val counts:  ", val_df["season_bin"].value_counts().sort_index().to_dict())
print("  test counts: ", test_df["season_bin"].value_counts().sort_index().to_dict())

Season regimes ready (0=winter, 1=shoulder, 2=summer)
  train counts: {0: 4076, 1: 4704, 2: 4881}
  val counts:   {0: 796, 1: 1100, 2: 1023}
  test counts:  {0: 858, 1: 916, 2: 885}


In [9]:
for d in (train_df, val_df, test_df):
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

stations = sorted(set(train_df["station_id"].dropna().unique())
                  | set(val_df["station_id"].dropna().unique())
                  | set(test_df["station_id"].dropna().unique()))

print("\n=== TEMPORAL LEAKAGE CHECKS (per station) ===")
bad = 0

for sid in stations:
    tr = train_df[train_df["station_id"] == sid]["date"].dropna()
    va = val_df[val_df["station_id"] == sid]["date"].dropna()
    te = test_df[test_df["station_id"] == sid]["date"].dropna()

    if len(tr) == 0 or len(va) == 0 or len(te) == 0:
        print(f"[WARN] station {sid}: missing split data (train={len(tr)}, val={len(va)}, test={len(te)})")
        bad += 1
        continue

    tr_min, tr_max = tr.min(), tr.max()
    va_min, va_max = va.min(), va.max()
    te_min, te_max = te.min(), te.max()

    # date overlap checks (hard leakage)
    overlap_tr_va = len(set(tr.unique()) & set(va.unique()))
    overlap_tr_te = len(set(tr.unique()) & set(te.unique()))
    overlap_va_te = len(set(va.unique()) & set(te.unique()))

    # ordering check (soft but important)
    order_ok = (tr_max < va_min) and (va_max < te_min)

    if overlap_tr_va or overlap_tr_te or overlap_va_te or (not order_ok):
        print(f"[ERROR] station {sid}:")
        print(f"  train: {tr_min} -> {tr_max}")
        print(f"  val:   {va_min} -> {va_max}")
        print(f"  test:  {te_min} -> {te_max}")
        print(f"  overlaps: train∩val={overlap_tr_va}, train∩test={overlap_tr_te}, val∩test={overlap_va_te}")
        print(f"  order_ok: {order_ok}")
        bad += 1
    else:
        print(f"[OK] station {sid}: train<{val_df is not None and 'val' or ''}val<test with no date overlap")

if bad == 0:
    print("\n[INFO] No temporal leakage detected.")
else:
    print(f"\n[WARNING] {bad} station(s) have temporal leakage or split issues.")



=== TEMPORAL LEAKAGE CHECKS (per station) ===
[OK] station Darrington: train<valval<test with no date overlap
[OK] station Quinault: train<valval<test with no date overlap
[OK] station SourdoughGulch_WA_985: train<valval<test with no date overlap
[OK] station Spokane: train<valval<test with no date overlap

[INFO] No temporal leakage detected.


In [10]:
# yoinked from v8.2

RAIN_COL = "precip_mm"
RAIN_THR = 4.0
K = 7
WEIGHTS = np.array([1.0, 0.6, 0.2, 0.1, 0.05, 0.02, 0.01, 0.0], dtype=float)  # len = K+1

NEW_FEATURES = [
    "rain_event_impulse_0_7",
    "rain_mm_impulse_0_7",
    "days_since_rain_event",
]

def add_rain_impulse_features(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.sort_values(["station_id", "date"]).reset_index(drop=True)

    d["_rain_event"] = (d[RAIN_COL] >= RAIN_THR).astype(int)

    for k in range(0, K + 1):
        d[f"_ev_lag{k}"] = d.groupby("station_id")["_rain_event"].shift(k)
        d[f"_mm_lag{k}"] = d.groupby("station_id")[RAIN_COL].shift(k)

    ev_cols = [f"_ev_lag{k}" for k in range(0, K + 1)]
    mm_cols = [f"_mm_lag{k}" for k in range(0, K + 1)]

    ev_mat = d[ev_cols].fillna(0).to_numpy(dtype=float)
    mm_mat = d[mm_cols].fillna(0).to_numpy(dtype=float)

    d["rain_event_impulse_0_7"] = (ev_mat * WEIGHTS).sum(axis=1)
    d["rain_mm_impulse_0_7"]    = (mm_mat * WEIGHTS).sum(axis=1)

    def _days_since_event(group: pd.DataFrame) -> pd.Series:
        ev = group["_rain_event"].to_numpy()
        out = np.full(len(ev), np.nan, dtype=float)
        last_idx = None
        for i in range(len(ev)):
            if ev[i] == 1:
                last_idx = i
                out[i] = 0.0
            else:
                if last_idx is not None:
                    out[i] = float(i - last_idx)
        return pd.Series(out, index=group.index)

    d["days_since_rain_event"] = (
        d.groupby("station_id", group_keys=False)
         .apply(_days_since_event)
         .clip(upper=30)
    )

    drop_cols = ["_rain_event"] + ev_cols + mm_cols
    d.drop(columns=drop_cols, inplace=True, errors="ignore")
    return d

train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()
train_df["_split"] = "train"
val_df["_split"]   = "val"
test_df["_split"]  = "test"

all_df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
all_df = add_rain_impulse_features(all_df)

train_df = all_df[all_df["_split"] == "train"].drop(columns=["_split"]).reset_index(drop=True)
val_df   = all_df[all_df["_split"] == "val"].drop(columns=["_split"]).reset_index(drop=True)
test_df  = all_df[all_df["_split"] == "test"].drop(columns=["_split"]).reset_index(drop=True)

FEATURE_COLS_A = list(FEATURE_COLS)
FEATURE_COLS_B = list(FEATURE_COLS) + NEW_FEATURES

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    miss = [c for c in NEW_FEATURES if c not in d.columns]
    if miss:
        raise ValueError(f"{name} missing NEW_FEATURES: {miss}")

print("v9.2 impulse features added (wet-expert-only)")
print("  baseline features (A):", len(FEATURE_COLS_A))
print("  wet expert features (B):", len(FEATURE_COLS_B))
print("  NaN rate (train/val/test):")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(" ", name, d[NEW_FEATURES].isna().mean().round(4).to_dict())

v9.2 impulse features added (wet-expert-only)
  baseline features (A): 46
  wet expert features (B): 49
  NaN rate (train/val/test):
  train {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0027}
  val {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0}
  test {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0}


In [11]:
print("\nTemporal split check (train -> validation):")

for sid in sorted(train_df["station_id"].unique()):
    train_dates = train_df.loc[train_df["station_id"] == sid, "date"]
    val_dates   = val_df.loc[val_df["station_id"] == sid, "date"]

    max_train = train_dates.max()
    min_val   = val_dates.min()

    print(f"  Station {sid}:")
    print(f"    train max date: {max_train}")
    print(f"    val   min date: {min_val}")


Temporal split check (train -> validation):
  Station Darrington:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Quinault:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station SourdoughGulch_WA_985:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Spokane:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00


## 5. Train / Validation / Test Split

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [12]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     13661
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2011-10-06 00:00:00 -- 2021-09-23 00:00:00

VAL
  rows:     2919
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-09-24 00:00:00 -- 2023-11-12 00:00:00

TEST
  rows:     2659
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2023-11-13 00:00:00 -- 2025-12-31 00:00:00

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


## 6. Model Definition

### 6.1 Feature Matrix Construction

In [33]:
A_train_df = train_df[train_df["season_bin"] != 0].copy()
A_val_df   = val_df[val_df["season_bin"] != 0].copy()
A_test_df  = test_df[test_df["season_bin"] != 0].copy()

XA_train = A_train_df[FEATURE_COLS_A].copy()
yA_train = A_train_df[TARGET_COL].to_numpy()

XA_val   = A_val_df[FEATURE_COLS_A].copy()
yA_val   = A_val_df[TARGET_COL].to_numpy()

XA_test  = A_test_df[FEATURE_COLS_A].copy()
yA_test  = A_test_df[TARGET_COL].to_numpy()

B_train_df = train_df[train_df["season_bin"].isin([0, 1])].copy()
B_val_df   = val_df[val_df["season_bin"] == 0].copy()
B_test_df  = test_df[test_df["season_bin"] == 0].copy()

XB_train = B_train_df[FEATURE_COLS_B].copy()
yB_train = B_train_df[TARGET_COL].to_numpy()

XB_val   = B_val_df[FEATURE_COLS_B].copy()
yB_val   = B_val_df[TARGET_COL].to_numpy()

XB_test  = B_test_df[FEATURE_COLS_B].copy()
yB_test  = B_test_df[TARGET_COL].to_numpy()

XB_val_full  = val_df[FEATURE_COLS_B].copy()
XB_test_full = test_df[FEATURE_COLS_B].copy()

print("v9.4-season matrices ready (B trains on winter+shoulder, eval winter-only)")
print("  Expert A (non-winter) train/val/test:", XA_train.shape, XA_val.shape, XA_test.shape)
print("  Expert B train (winter+shoulder):", XB_train.shape)
print("  Expert B eval winter val/test:", XB_val.shape, XB_test.shape)

print("  Season counts (train):", train_df["season_bin"].value_counts().sort_index().to_dict())
print("  A train seasons:", A_train_df["season_bin"].value_counts().sort_index().to_dict())
print("  B train seasons:", B_train_df["season_bin"].value_counts().sort_index().to_dict())
print("  B val seasons:", B_val_df["season_bin"].value_counts().sort_index().to_dict())
print("  B test seasons:", B_test_df["season_bin"].value_counts().sort_index().to_dict())

W_WINTER = 2.5
W_SHOULDER = 1.0
wB_train = np.where(B_train_df["season_bin"].to_numpy() == 0, W_WINTER, W_SHOULDER).astype(float)
print("  wB_train mean/min/max:", round(float(wB_train.mean()), 3), float(wB_train.min()), float(wB_train.max()))

v9.4-season matrices ready (B trains on winter+shoulder, eval winter-only)
  Expert A (non-winter) train/val/test: (9585, 46) (2123, 46) (1801, 46)
  Expert B train (winter+shoulder): (8780, 49)
  Expert B eval winter val/test: (796, 49) (858, 49)
  Season counts (train): {0: 4076, 1: 4704, 2: 4881}
  A train seasons: {1: 4704, 2: 4881}
  B train seasons: {0: 4076, 1: 4704}
  B val seasons: {0: 796}
  B test seasons: {0: 858}
  wB_train mean/min/max: 1.696 1.0 2.5


In [34]:
# winter = season_bin == 0
# gate outputs 1 for winter (use Expert B), 0 otherwise (use Expert A)

gate_val  = (val_df["season_bin"].to_numpy() == 0).astype(int)
gate_test = (test_df["season_bin"].to_numpy() == 0).astype(int)

print("Season gate ready")
print("  winter% val/test:",
      gate_val.mean().round(3),
      gate_test.mean().round(3))

Season gate ready
  winter% val/test: 0.273 0.323


In [35]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

### 6.4 Expert A (Model A) | DRY

In [36]:
xgbA = XGBRegressor(
    subsample=0.9,
    reg_lambda=2.0,
    reg_alpha=0.05,
    n_estimators=4000,
    min_child_weight=3,
    max_depth=7,
    learning_rate=0.05,
    gamma=0.0,
    colsample_bytree=0.75,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)

rfA = RandomForestRegressor(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    max_depth=16,
    n_jobs=-1,
    random_state=42,
)

xgbA.fit(XA_train, yA_train)
rfA.fit(XA_train, yA_train)

# meta learner (ridge) trained on base preds
A_pred_train = np.vstack([xgbA.predict(XA_train), rfA.predict(XA_train)]).T
A_pred_val   = np.vstack([xgbA.predict(XA_val),   rfA.predict(XA_val)]).T

ridgeA = Ridge(alpha=1.0, random_state=42)
ridgeA.fit(A_pred_train, yA_train)

# evaluate on NON-WINTER val
yA_hat_val = ridgeA.predict(A_pred_val)

r2_A  = r2_score(yA_val, yA_hat_val)
mae_A = mean_absolute_error(yA_val, yA_hat_val)
rmse_A = np.sqrt(mean_squared_error(yA_val, yA_hat_val))

print("Expert A (NON-WINTER) trained")
print("  val metrics (NON-WINTER only):")
print(f"    R2  : {r2_A:.6f}")
print(f"    MAE : {mae_A:.6f}")
print(f"    RMSE: {rmse_A:.6f}")
print("  ridge weights:", ridgeA.coef_.round(6), "intercept:", float(ridgeA.intercept_))

Expert A (NON-WINTER) trained
  val metrics (NON-WINTER only):
    R2  : 0.802809
    MAE : 0.033708
    RMSE: 0.045191
  ridge weights: [0.690472 0.314446] intercept: -0.0008637693099084742


In [37]:
XA_val_full  = val_df[FEATURE_COLS_A].copy()
XA_test_full = test_df[FEATURE_COLS_A].copy()

A_base_val  = np.vstack([
    xgbA.predict(XA_val_full),
    rfA.predict(XA_val_full)
]).T

A_base_test = np.vstack([
    xgbA.predict(XA_test_full),
    rfA.predict(XA_test_full)
]).T

yhatA_val  = ridgeA.predict(A_base_val)
yhatA_test = ridgeA.predict(A_base_test)

print("Expert A full-split predictions ready (NON-WINTER expert)")
print("  yhatA_val :", yhatA_val.shape,
      "min/mean/max:", float(np.min(yhatA_val)), float(np.mean(yhatA_val)), float(np.max(yhatA_val)))
print("  yhatA_test:", yhatA_test.shape,
      "min/mean/max:", float(np.min(yhatA_test)), float(np.mean(yhatA_test)), float(np.max(yhatA_test)))

Expert A full-split predictions ready (NON-WINTER expert)
  yhatA_val : (2919,) min/mean/max: 0.016375330454175496 0.20054569678300377 0.349147025821548
  yhatA_test: (2659,) min/mean/max: 0.013098920674991969 0.204634240844234 0.3411256878792419


In [38]:
A_base_val_slice  = np.vstack([xgbA.predict(XA_val),  rfA.predict(XA_val)]).T
A_base_test_slice = np.vstack([xgbA.predict(XA_test), rfA.predict(XA_test)]).T

yhatA_val_slice  = ridgeA.predict(A_base_val_slice)
yhatA_test_slice = ridgeA.predict(A_base_test_slice)

r2v_A, maev_A, rmsev_A = _metrics(yA_val, yhatA_val_slice)
r2t_A, maet_A, rmset_A = _metrics(yA_test, yhatA_test_slice)

print("Expert A (NON-WINTER) sanity on NON-WINTER slices:")
print(f"  VAL : R2={r2v_A:.6f} MAE={maev_A:.6f} RMSE={rmsev_A:.6f}  (n={len(yA_val)})")
print(f"  TEST: R2={r2t_A:.6f} MAE={maet_A:.6f} RMSE={rmset_A:.6f}  (n={len(yA_test)})")

Expert A (NON-WINTER) sanity on NON-WINTER slices:
  VAL : R2=0.802809 MAE=0.033708 RMSE=0.045191  (n=2123)
  TEST: R2=0.748230 MAE=0.035464 RMSE=0.046443  (n=1801)


### 6.5 Expert B (Model B) | WET

In [41]:
xgbB = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=8,
    subsample=0.85,
    colsample_bytree=0.8,
    reg_lambda=6.0,
    reg_alpha=0.1,
    gamma=0.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgbB.fit(XB_train, yB_train)

yhatB_val_slice  = xgbB.predict(XB_val)
yhatB_test_slice = xgbB.predict(XB_test)

r2v_w, maev_w, rmsev_w = _metrics(yB_val,  yhatB_val_slice)
r2t_w, maet_w, rmset_w = _metrics(yB_test, yhatB_test_slice)

In [42]:
print("=== Expert B (WINTER) performance on WINTER slices ===")
print(f"  VAL  winter : R2={r2v_w:.6f}  MAE={maev_w:.6f}  RMSE={rmsev_w:.6f}  (n={len(yB_val)})")
print(f"  TEST winter : R2={r2t_w:.6f}  MAE={maet_w:.6f}  RMSE={rmset_w:.6f}  (n={len(yB_test)})")

yhatB_val  = xgbB.predict(XB_val_full)
yhatB_test = xgbB.predict(XB_test_full)

print("\nExpert B full-split predictions ready")
print("  yhatB_val :", yhatB_val.shape,  "min/mean/max:",
      round(float(np.min(yhatB_val)), 6), round(float(np.mean(yhatB_val)), 6), round(float(np.max(yhatB_val)), 6))
print("  yhatB_test:", yhatB_test.shape, "min/mean/max:",
      round(float(np.min(yhatB_test)), 6), round(float(np.mean(yhatB_test)), 6), round(float(np.max(yhatB_test)), 6))

y_val  = val_df[TARGET_COL].to_numpy()
y_test = test_df[TARGET_COL].to_numpy()

r2v_full, maev_full, rmsev_full = _metrics(y_val,  yhatB_val)
r2t_full, maet_full, rmset_full = _metrics(y_test, yhatB_test)

print("\n=== Expert B sanity on FULL splits (not gated) ===")
print(f"  VAL  full : R2={r2v_full:.6f}  MAE={maev_full:.6f}  RMSE={rmsev_full:.6f}")
print(f"  TEST full : R2={r2t_full:.6f}  MAE={maet_full:.6f}  RMSE={rmset_full:.6f}")

=== Expert B (WINTER) performance on WINTER slices ===
  VAL  winter : R2=0.049053  MAE=0.028298  RMSE=0.037333  (n=796)
  TEST winter : R2=-0.501590  MAE=0.042395  RMSE=0.055942  (n=858)

Expert B full-split predictions ready
  yhatB_val : (2919,) min/mean/max: 0.02822 0.220226 0.359143
  yhatB_test: (2659,) min/mean/max: 0.020476 0.215352 0.352045

=== Expert B sanity on FULL splits (not gated) ===
  VAL  full : R2=0.811838  MAE=0.034621  RMSE=0.043689
  TEST full : R2=0.673233  MAE=0.043578  RMSE=0.054914


### 6.6 Residual Model

In [45]:
BASE_FEATURES = [
    "DOY",
    "G_API",
    "G_rain_sum_30d",
    "C_lag_LST_modis_kobs30",
]

RESIDUAL_FEATURES = [
    "rain_event_impulse_0_7",
    "rain_mm_impulse_0_7",
    "days_since_rain_event",

    "E_SAR_diff",
    "C_lag_E_SAR_diff_kobs6",

    "A_d_LST_modis_kobs7",
]

In [46]:
RESIDUAL_FEATURES2 = list(dict.fromkeys(RESIDUAL_FEATURES))

need_cols = list(dict.fromkeys(BASE_FEATURES + RESIDUAL_FEATURES2 + [TARGET_COL]))
missing = [c for c in need_cols if c not in B_train_df.columns]
if missing:
    raise ValueError(f"Missing columns in B_train_df: {missing}")

# ---- baseline X (winter slices) ----
Xb_base_train = B_train_df[BASE_FEATURES].copy()
Xb_base_val   = B_val_df[BASE_FEATURES].copy()
Xb_base_test  = B_test_df[BASE_FEATURES].copy()

# ---- baseline y ----
yB_base_train = yB_train
yB_base_val   = yB_val
yB_base_test  = yB_test

baseline = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("ridge", Ridge(alpha=5.0)),
])

baseline.fit(Xb_base_train, yB_base_train)

y_base_train = baseline.predict(Xb_base_train)
y_base_val   = baseline.predict(Xb_base_val)
y_base_test  = baseline.predict(Xb_base_test)

In [ ]:
res_train = yB_train - y_base_train
res_val   = yB_val   - y_base_val
res_test  = yB_test  - y_base_test

XR2_train = B_train_df[RESIDUAL_FEATURES2].copy()
XR2_val   = B_val_df[RESIDUAL_FEATURES2].copy()
XR2_test  = B_test_df[RESIDUAL_FEATURES2].copy()

print("Residual setup ready (WINTER only)")
print("  BASE_FEATURES:", BASE_FEATURES)
print("  RESIDUAL_FEATURES2 (extra-only):", RESIDUAL_FEATURES2)
print("  baseline X shapes (train/val/test):", Xb_base_train.shape, Xb_base_val.shape, Xb_base_test.shape)
print("  residual X shapes (train/val/test):", XR2_train.shape, XR2_val.shape, XR2_test.shape)

print("  residual std (train/val/test):",
      round(float(np.std(res_train)), 6),
      round(float(np.std(res_val)), 6),
      round(float(np.std(res_test)), 6))

print("  NaN rate mean residual X (train/val/test):",
      round(float(XR2_train.isna().mean().mean()), 4),
      round(float(XR2_val.isna().mean().mean()), 4),
      round(float(XR2_test.isna().mean().mean()), 4))

nan_by_feat = XR2_train.isna().mean().sort_values(ascending=False)
print("  top NaN residual features (train):", nan_by_feat.head(6).round(3).to_dict())

Residual setup ready (WINTER only)
  BASE_FEATURES: ['DOY', 'G_API', 'G_rain_sum_30d', 'C_lag_LST_modis_kobs30']
  RESIDUAL_FEATURES2 (extra-only): ['rain_event_impulse_0_7', 'rain_mm_impulse_0_7', 'days_since_rain_event', 'E_SAR_diff', 'C_lag_E_SAR_diff_kobs6', 'A_d_LST_modis_kobs7']
  baseline X shapes (train/val/test): (8780, 4) (796, 4) (858, 4)
  residual X shapes (train/val/test): (8780, 6) (796, 6) (858, 6)
  residual std (train/val/test): 0.066669 0.040301 0.051786
  NaN rate mean residual X (train/val/test): 0.0017 0.0 0.0
  top NaN residual features (train): {'days_since_rain_event': 0.004, 'A_d_LST_modis_kobs7': 0.003, 'C_lag_E_SAR_diff_kobs6': 0.003, 'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'E_SAR_diff': 0.0}


In [48]:
xgb_res = XGBRegressor(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10.0,
    reg_alpha=0.2,
    gamma=0.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_res.fit(XR2_train, res_train)

res_hat_val  = xgb_res.predict(XR2_val)
res_hat_test = xgb_res.predict(XR2_test)

r2_res_val, mae_res_val, rmse_res_val = _metrics(res_val, res_hat_val)
corr_res_val = float(np.corrcoef(res_val, res_hat_val)[0, 1])

In [49]:
print("Residual model sanity (WINTER slices):")
print(f"  R2(residual) VAL: {r2_res_val:.6f}   corr(res, res_hat) VAL: {corr_res_val:.6f}")

offset = float(np.mean(res_hat_val))
res_hat_val  = res_hat_val  - offset
res_hat_test = res_hat_test - offset
print("recentering offset:", round(offset, 6))

RES_CLIP = 0.12
raw_val, raw_test = res_hat_val.copy(), res_hat_test.copy()

res_hat_val  = np.clip(res_hat_val,  -RES_CLIP, RES_CLIP)
res_hat_test = np.clip(res_hat_test, -RES_CLIP, RES_CLIP)

print("clip fraction val/test:",
      round(float(np.mean(np.abs(raw_val)  > RES_CLIP)), 3),
      round(float(np.mean(np.abs(raw_test) > RES_CLIP)), 3))

Residual model sanity (WINTER slices):
  R2(residual) VAL: -0.433287   corr(res, res_hat) VAL: 0.118761
recentering offset: 0.006359
clip fraction val/test: 0.0 0.0


In [50]:
yhatB_val_slice  = y_base_val  + res_hat_val
yhatB_test_slice = y_base_test + res_hat_test

r2v_base, maev_base, rmsev_base = _metrics(yB_val,  y_base_val)
r2t_base, maet_base, rmset_base = _metrics(yB_test, y_base_test)

r2v_final, maev_final, rmsev_final = _metrics(yB_val,  yhatB_val_slice)
r2t_final, maet_final, rmset_final = _metrics(yB_test, yhatB_test_slice)

print("\n=== Expert B (WINTER slices) — Baseline vs Baseline+Residual ===")
print("Baseline only:")
print(f"  VAL  winter : R2={r2v_base:.6f} MAE={maev_base:.6f} RMSE={rmsev_base:.6f}  (n={len(yB_val)})")
print(f"  TEST winter : R2={r2t_base:.6f} MAE={maet_base:.6f} RMSE={rmset_base:.6f}  (n={len(yB_test)})")

print("Baseline + residual:")
print(f"  VAL  winter : R2={r2v_final:.6f} MAE={maev_final:.6f} RMSE={rmsev_final:.6f}  (n={len(yB_val)})")
print(f"  TEST winter : R2={r2t_final:.6f} MAE={maet_final:.6f} RMSE={rmset_final:.6f}  (n={len(yB_test)})")

print("Lift from residual (positive is good):")
print(f"  VAL  ΔR2 = {r2v_final - r2v_base:+.6f}   ΔRMSE = {rmsev_base - rmsev_final:+.6f}")
print(f"  TEST ΔR2 = {r2t_final - r2t_base:+.6f}   ΔRMSE = {rmset_base - rmset_final:+.6f}")


=== Expert B (WINTER slices) — Baseline vs Baseline+Residual ===
Baseline only:
  VAL  winter : R2=-0.192740 MAE=0.031958 RMSE=0.041810  (n=796)
  TEST winter : R2=-0.341004 MAE=0.038351 RMSE=0.052866  (n=858)
Baseline + residual:
  VAL  winter : R2=-0.657364 MAE=0.037551 RMSE=0.049286  (n=796)
  TEST winter : R2=-0.939463 MAE=0.047383 RMSE=0.063577  (n=858)
Lift from residual (positive is good):
  VAL  ΔR2 = -0.464624   ΔRMSE = -0.007475
  TEST ΔR2 = -0.598459   ΔRMSE = -0.010711


In [57]:
XB_val_base_full  = val_df[BASE_FEATURES].copy()
XB_test_base_full = test_df[BASE_FEATURES].copy()

y_base_val_full  = baseline.predict(XB_val_base_full)
y_base_test_full = baseline.predict(XB_test_base_full)

XR2_val_full  = val_df[RESIDUAL_FEATURES2].copy()
XR2_test_full = test_df[RESIDUAL_FEATURES2].copy()

res_hat_val_full  = xgb_res.predict(XR2_val_full)
res_hat_test_full = xgb_res.predict(XR2_test_full)

res_hat_val_full  = np.clip(res_hat_val_full  - offset, -RES_CLIP, RES_CLIP)
res_hat_test_full = np.clip(res_hat_test_full - offset, -RES_CLIP, RES_CLIP)

yhatB_val  = y_base_val_full  + res_hat_val_full
yhatB_test = y_base_test_full + res_hat_test_full

print("Expert B full-split predictions ready (WINTER expert = baseline + residual)")
print("  yhatB_val :", yhatB_val.shape,  "min/mean/max:",
      float(np.min(yhatB_val)), float(np.mean(yhatB_val)), float(np.max(yhatB_val)))
print("  yhatB_test:", yhatB_test.shape, "min/mean/max:",
      float(np.min(yhatB_test)), float(np.mean(yhatB_test)), float(np.max(yhatB_test)))

Expert B full-split predictions ready (WINTER expert = baseline + residual)
  yhatB_val : (2919,) min/mean/max: 0.020202484412348842 0.21278746992796177 0.38638502107171313
  yhatB_test: (2659,) min/mean/max: 0.0310463935652473 0.21628962876597302 0.4278208184204691


In [ ]:

XB_val_winter  = B_val_df[BASE_FEATURES].copy()
XB_test_winter = B_test_df[BASE_FEATURES].copy()

yhatB_val_winter  = baseline.predict(XB_val_winter)
yhatB_test_winter = baseline.predict(XB_test_winter)

r2v, maev, rmsev = _metrics(yB_val,  yhatB_val_winter)
r2t, maet, rmset = _metrics(yB_test, yhatB_test_winter)

print("Expert B (WINTER) on WINTER-only slices (baseline-only Ridge):")
print("  XB_val_winter:", XB_val_winter.shape, "yB_val:", yB_val.shape)
print("  XB_test_winter:", XB_test_winter.shape, "yB_test:", yB_test.shape)
print(f"  VAL  winter : R2={r2v:.6f} MAE={maev:.6f} RMSE={rmsev:.6f}  (n={len(yB_val)})")
print(f"  TEST winter : R2={r2t:.6f} MAE={maet:.6f} RMSE={rmset:.6f}  (n={len(yB_test)})")

Expert B (WINTER) on WINTER-only slices (baseline-only Ridge):
  VAL  winter : R2=-0.192740 MAE=0.031958 RMSE=0.041810  (n=796)
  TEST winter : R2=-0.341004 MAE=0.038351 RMSE=0.052866  (n=858)


## 7. Gating

In [63]:
gate_val_winter  = (val_df["season_bin"].to_numpy() == 0).astype(int)
gate_test_winter = (test_df["season_bin"].to_numpy() == 0).astype(int)

yhat_mix_val  = gate_val_winter  * yhatB_val  + (1 - gate_val_winter)  * yhatA_val
yhat_mix_test = gate_test_winter * yhatB_test + (1 - gate_test_winter) * yhatA_test

print("Mixture predictions ready (hard season gate)")
print("  VAL  mix :", yhat_mix_val.shape)
print("  TEST mix:", yhat_mix_test.shape)
print("  % winter routed (val/test):",
      gate_val_winter.mean().round(3),
      gate_test_winter.mean().round(3))

Mixture predictions ready (hard season gate)
  VAL  mix : (2919,)
  TEST mix: (2659,)
  % winter routed (val/test): 0.273 0.323


In [60]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def _print_row(name, y_true, y_pred):
    r2, mae, rmse = _metrics(y_true, y_pred)
    print(f"{name:<22} R2={r2: .6f}  MAE={mae: .6f}  RMSE={rmse: .6f}")

y_val  = val_df[TARGET_COL].to_numpy()
y_test = test_df[TARGET_COL].to_numpy()

val_is_winter  = (val_df["season_bin"].to_numpy() == 0)
test_is_winter = (test_df["season_bin"].to_numpy() == 0)

print("=== OVERALL (FULL SPLITS) ===")
_print_row("Expert A only (VAL)", y_val, yhatA_val)
_print_row("Expert B only (VAL)", y_val, yhatB_val)
_print_row("MIX (VAL)",           y_val, yhat_mix_val)
print()
_print_row("Expert A only (TEST)", y_test, yhatA_test)
_print_row("Expert B only (TEST)", y_test, yhatB_test)
_print_row("MIX (TEST)",           y_test, yhat_mix_test)

print("\n=== WINTER-ONLY SLICES ===")
_print_row("Expert A on winter (VAL)", y_val[val_is_winter],  yhatA_val[val_is_winter])
_print_row("Expert B on winter (VAL)", y_val[val_is_winter],  yhatB_val[val_is_winter])
_print_row("MIX on winter (VAL)",      y_val[val_is_winter],  yhat_mix_val[val_is_winter])
print()
_print_row("Expert A on winter (TEST)", y_test[test_is_winter], yhatA_test[test_is_winter])
_print_row("Expert B on winter (TEST)", y_test[test_is_winter], yhatB_test[test_is_winter])
_print_row("MIX on winter (TEST)",      y_test[test_is_winter], yhat_mix_test[test_is_winter])

print("\n=== NON-WINTER SLICES ===")
_print_row("Expert A on non-winter (VAL)", y_val[~val_is_winter],  yhatA_val[~val_is_winter])
_print_row("Expert B on non-winter (VAL)", y_val[~val_is_winter],  yhatB_val[~val_is_winter])
_print_row("MIX on non-winter (VAL)",      y_val[~val_is_winter],  yhat_mix_val[~val_is_winter])
print()
_print_row("Expert A on non-winter (TEST)", y_test[~test_is_winter], yhatA_test[~test_is_winter])
_print_row("Expert B on non-winter (TEST)", y_test[~test_is_winter], yhatB_test[~test_is_winter])
_print_row("MIX on non-winter (TEST)",      y_test[~test_is_winter], yhat_mix_test[~test_is_winter])

print("\nCounts (val/test):")
print("  winter:", int(val_is_winter.sum()), int(test_is_winter.sum()))
print("  non-winter:", int((~val_is_winter).sum()), int((~test_is_winter).sum()))

=== OVERALL (FULL SPLITS) ===
Expert A only (VAL)    R2= 0.796798  MAE= 0.033750  RMSE= 0.045402
Expert B only (VAL)    R2= 0.660583  MAE= 0.046793  RMSE= 0.058678
MIX (VAL)              R2= 0.788282  MAE= 0.034756  RMSE= 0.046343

Expert A only (TEST)   R2= 0.738779  MAE= 0.036568  RMSE= 0.049098
Expert B only (TEST)   R2= 0.604231  MAE= 0.048082  RMSE= 0.060434
MIX (TEST)             R2= 0.700360  MAE= 0.039310  RMSE= 0.052585

=== WINTER-ONLY SLICES ===
Expert A on winter (VAL) R2=-0.441218  MAE= 0.033860  RMSE= 0.045960
Expert B on winter (VAL) R2=-0.657364  MAE= 0.037551  RMSE= 0.049286
MIX on winter (VAL)    R2=-0.657364  MAE= 0.037551  RMSE= 0.049286

Expert A on winter (TEST) R2=-0.412251  MAE= 0.038886  RMSE= 0.054252
Expert B on winter (TEST) R2=-0.939463  MAE= 0.047383  RMSE= 0.063577
MIX on winter (TEST)   R2=-0.939463  MAE= 0.047383  RMSE= 0.063577

=== NON-WINTER SLICES ===
Expert A on non-winter (VAL) R2= 0.802809  MAE= 0.033708  RMSE= 0.045191
Expert B on non-winter (VA

In [84]:
GATE_K = 3.8
CAP = 0.35

doy_train = train_df["DOY"].to_numpy(dtype=float)
doy_val   = val_df["DOY"].to_numpy(dtype=float)
doy_test  = test_df["DOY"].to_numpy(dtype=float)

# theta_train = 2.0 * np.pi * (doy_train / 365.25)
# theta_val   = 2.0 * np.pi * (doy_val   / 365.25)
# theta_test  = 2.0 * np.pi * (doy_test  / 365.25)

# score_train = np.cos(theta_train)
# score_val   = np.cos(theta_val)
# score_test  = np.cos(theta_test)

winter_center = 355.0
theta_train = 2.0 * np.pi * ((doy_train - winter_center) / 365.25)
theta_val   = 2.0 * np.pi * ((doy_val   - winter_center) / 365.25)
theta_test  = 2.0 * np.pi * ((doy_test  - winter_center) / 365.25)

# score_train = np.cos(theta_train)
# score_val   = np.cos(theta_val)
# score_test  = np.cos(theta_test)

score_train = np.cos(theta_train) * (1.0 - np.abs(np.sin(theta_train)))
score_val   = np.cos(theta_val)   * (1.0 - np.abs(np.sin(theta_val)))
score_test  = np.cos(theta_test)  * (1.0 - np.abs(np.sin(theta_test)))

winter_rate_train = (train_df["season_bin"].to_numpy() == 0).mean()
thr = np.quantile(score_train, 1.0 - winter_rate_train)

wB_val  = CAP * expit(GATE_K * (score_val  - thr))
wB_test = CAP * expit(GATE_K * (score_test - thr))

wB_val  = np.minimum(wB_val, 0.3)
wB_test = np.minimum(wB_test, 0.3)

yhat_soft_val  = wB_val  * yhatB_val  + (1.0 - wB_val)  * yhatA_val
yhat_soft_test = wB_test * yhatB_test + (1.0 - wB_test) * yhatA_test

print("Soft season gate ready (DOY-based, train-calibrated, capped)")
print("  winter_rate_train:", round(float(winter_rate_train), 3))
print("  thr:", round(float(thr), 6))
print("  CAP:", CAP, "GATE_K:", GATE_K)
print("  wB val  mean/min/max:",
      round(float(wB_val.mean()), 3),
      round(float(wB_val.min()), 3),
      round(float(wB_val.max()), 3))
print("  wB test mean/min/max:",
      round(float(wB_test.mean()), 3),
      round(float(wB_test.min()), 3),
      round(float(wB_test.max()), 3))
print("  frac wB>0.10 val/test:",
      round(float((wB_val  > 0.10).mean()), 3),
      round(float((wB_test > 0.10).mean()), 3))
print("  frac wB>0.30 val/test:",
      round(float((wB_val  > 0.30).mean()), 3),
      round(float((wB_test > 0.30).mean()), 3))

errA = np.abs(y_val - yhatA_val)
errB = np.abs(y_val - yhatB_val)

print("Mean |errA| vs |errB| when wB>0.3 (VAL):",
      errA[wB_val > 0.3].mean(),
      errB[wB_val > 0.3].mean())

Soft season gate ready (DOY-based, train-calibrated, capped)
  winter_rate_train: 0.298
  thr: 0.077704
  CAP: 0.35 GATE_K: 3.8
  wB val  mean/min/max: 0.15 0.006 0.3
  wB test mean/min/max: 0.154 0.006 0.3
  frac wB>0.10 val/test: 0.729 0.721
  frac wB>0.30 val/test: 0.0 0.0
Mean |errA| vs |errB| when wB>0.3 (VAL): nan nan


In [85]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def _print(name, y_true, y_pred):
    r2, mae, rmse = _metrics(y_true, y_pred)
    print(f"{name:<22} R2={r2: .6f}  MAE={mae: .6f}  RMSE={rmse: .6f}")

y_val  = val_df[TARGET_COL].to_numpy()
y_test = test_df[TARGET_COL].to_numpy()

val_is_winter  = (val_df["season_bin"].to_numpy() == 0)
test_is_winter = (test_df["season_bin"].to_numpy() == 0)

print("=== OVERALL ===")
_print("Hard MIX (VAL)", y_val, yhat_mix_val)
_print("Soft MIX (VAL)", y_val, yhat_soft_val)
print()
_print("Hard MIX (TEST)", y_test, yhat_mix_test)
_print("Soft MIX (TEST)", y_test, yhat_soft_test)

print("\n=== WINTER ONLY ===")
_print("Hard MIX winter (VAL)", y_val[val_is_winter], yhat_mix_val[val_is_winter])
_print("Soft MIX winter (VAL)", y_val[val_is_winter], yhat_soft_val[val_is_winter])
print()
_print("Hard MIX winter (TEST)", y_test[test_is_winter], yhat_mix_test[test_is_winter])
_print("Soft MIX winter (TEST)", y_test[test_is_winter], yhat_soft_test[test_is_winter])

print("\n=== NON-WINTER ===")
_print("Hard MIX non-winter (VAL)", y_val[~val_is_winter], yhat_mix_val[~val_is_winter])
_print("Soft MIX non-winter (VAL)", y_val[~val_is_winter], yhat_soft_val[~val_is_winter])
print()
_print("Hard MIX non-winter (TEST)", y_test[~test_is_winter], yhat_mix_test[~test_is_winter])
_print("Soft MIX non-winter (TEST)", y_test[~test_is_winter], yhat_soft_test[~test_is_winter])

=== OVERALL ===
Hard MIX (VAL)         R2= 0.788282  MAE= 0.034756  RMSE= 0.046343
Soft MIX (VAL)         R2= 0.807040  MAE= 0.033274  RMSE= 0.044243

Hard MIX (TEST)        R2= 0.700360  MAE= 0.039310  RMSE= 0.052585
Soft MIX (TEST)        R2= 0.743106  MAE= 0.036480  RMSE= 0.048690

=== WINTER ONLY ===
Hard MIX winter (VAL)  R2=-0.657364  MAE= 0.037551  RMSE= 0.049286
Soft MIX winter (VAL)  R2=-0.289769  MAE= 0.031671  RMSE= 0.043478

Hard MIX winter (TEST) R2=-0.939463  MAE= 0.047383  RMSE= 0.063577
Soft MIX winter (TEST) R2=-0.423605  MAE= 0.038878  RMSE= 0.054469

=== NON-WINTER ===
Hard MIX non-winter (VAL) R2= 0.802809  MAE= 0.033708  RMSE= 0.045191
Soft MIX non-winter (VAL) R2= 0.808566  MAE= 0.033876  RMSE= 0.044526

Hard MIX non-winter (TEST) R2= 0.748230  MAE= 0.035464  RMSE= 0.046443
Soft MIX non-winter (TEST) R2= 0.756427  MAE= 0.035337  RMSE= 0.045680


---

_Jakob Balkovec_